# ADS-B Collector — Schritt für Schritt

Dieses Notebook erklärt den Collector und führt ihn aus.

**Architektur:**
```
adsb.lol API  →  fetch_berlin()  →  build_document()  →  MongoDB Atlas (airline_landing.adsb_raw)
```

**Voraussetzung:** `.env` im **Projekt-Root** (`airline-data-platform/.env`) mit:
```
MONGO_URI=mongodb+srv://airline-collector-rw:<password>@mongo-mk1.ptb1k2b.mongodb.net/airline_landing
MONGO_DB=airline_landing
```
Dieser Collector schreibt Daten — `MONGO_URI` muss auf `airline-collector-rw` (Schreibzugriff) zeigen.
Credentials: `.env` ist gitignored, nie committen.

## Schritt 1 — Imports und Pfad-Setup

Der Collector liegt in `collectors/adsb_collector.py`.  
Der MongoDB-Connector liegt in `db/mongo/connector.py`.  
Damit Python diese Module findet, muss das aktuelle Verzeichnis (`03-data-collection/`) im Suchpfad sein.

In [ ]:
import sys
from pathlib import Path

# Sicherstellen, dass 03-data-collection/ im Suchpfad ist
notebook_dir = Path(".").resolve()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

print(f"Arbeitsverzeichnis: {notebook_dir}")

## Schritt 2 — API direkt abfragen

`fetch_berlin()` macht einen HTTP GET an `adsb.lol` und gibt die rohe API-Antwort zurück.  
Kein Login nötig — die API ist öffentlich.

In [ ]:
from collectors.adsb_collector import fetch_berlin

raw = fetch_berlin()

print(f"API-Timestamp (ms): {raw['now']}")
print(f"Flugzeuge gefunden: {raw['total']}")
print(f"Verarbeitungszeit:  {raw.get('ptime')} ms")
print(f"\nErste 2 Einträge in 'ac':")
for ac in raw['ac'][:2]:
    print(ac)

## Schritt 3 — Rohdaten als DataFrame anschauen

Das `ac`-Array enthält ein Dict pro Flugzeug. Wichtige Felder:

| Feld | Bedeutung |
|---|---|
| `hex` | ICAO-Adresse (eindeutige Kennung des Transponders) |
| `flight` | Callsign (z.B. `DLH123`) |
| `lat` / `lon` | Position |
| `alt_baro` | Barometrische Höhe in Fuß, oder `"ground"` |
| `gs` | Groundspeed in Knoten |
| `r` | Registrierung (z.B. `D-AIZW`) |
| `t` | Flugzeugtyp (z.B. `A320`) |

In [ ]:
import pandas as pd

df = pd.DataFrame(raw['ac'])
print(f"Shape: {df.shape}  ({df.shape[0]} Flugzeuge, {df.shape[1]} Felder)")
print(f"Alle Felder: {list(df.columns)}\n")

show = [c for c in ['hex', 'flight', 'r', 't', 'lat', 'lon', 'alt_baro', 'gs'] if c in df.columns]
df[show].head(10)

## Schritt 4 — Dokument bauen

`build_document()` verpackt die Rohantwort in das MongoDB-Format.  
Das `ac`-Array bleibt **unverändert** — keine Transformation, keine Bereinigung.  
Das ist bewusst: MongoDB ist die Raw Landing Zone.

In [ ]:
from collectors.adsb_collector import build_document

doc = build_document(raw)

# Alles außer dem ac-Array anzeigen
meta = {k: v for k, v in doc.items() if k != 'ac'}
for k, v in meta.items():
    print(f"  {k}: {v}")
print(f"  ac: [{len(doc['ac'])} Flugzeug-Dicts]")

## Schritt 5 — Mit MongoDB verbinden

`from_env()` liest `MONGO_URI` und `MONGO_DB` aus der `.env`-Datei.  
Der Context Manager (`with`) stellt sicher, dass die Verbindung am Ende immer geschlossen wird.

In [ ]:
from db.mongo.connector import from_env

db = from_env().connect()
print("Verbunden.")

## Schritt 6 — Snapshot in MongoDB schreiben

`insert_adsb_snapshot()` ruft intern `collection.insert_one(doc)` auf.  
MongoDB vergibt automatisch eine `_id` (ObjectId) und gibt sie zurück.

In [ ]:
inserted_id = db.insert_adsb_snapshot("adsb_raw", doc)
print(f"Gespeichert mit _id: {inserted_id}")

total = db.collection("adsb_raw").count_documents({})
print(f"Snapshots gesamt in MongoDB: {total}")

## Schritt 7 — Mehrere Snapshots sammeln (Mini-Loop)

Für Tests: 3 Snapshots im Abstand von 30 Sekunden sammeln.  
In der Praxis läuft der Collector als Script: `python collectors/adsb_collector.py --interval 60`

In [ ]:
import time

N = 3          # Anzahl Snapshots
INTERVAL = 30  # Sekunden zwischen den Abfragen

for i in range(N):
    raw_i = fetch_berlin()
    doc_i = build_document(raw_i)
    _id = db.insert_adsb_snapshot("adsb_raw", doc_i)
    print(f"[{i+1}/{N}] {doc_i['collected_at']}  —  {doc_i['total']} aircraft  →  _id={_id}")
    if i < N - 1:
        time.sleep(INTERVAL)

print("\nFertig.")

## Schritt 8 — Verbindung schließen

In [ ]:
db.close()
print("Verbindung geschlossen.")